# Polygenic score calculation using PRS-CS

## Notes before running the scripts
- Remember to replace all data and working directory paths with your actual paths.
- Depedencies: plink2, remember to set the path of tools with your actual paths
- download LD reference at https://github.com/getian107/PRScs or https://personal.broadinstitute.org/hhuang/public/PRS-CSx/Reference/UKBB/ldblk_ukbb_eur.tar.gz


In [ ]:
import pandas as pd
import subprocess
import tempfile
import os
import re
from mpire import WorkerPool
from io import StringIO
import numpy as np
n_threads = "10" # change n_threads value with your computational resource, n_cpus = n_threads * njobs
os.environ["OMP_NUM_THREADS"] = n_threads
os.environ["MKL_NUM_THREADS"] = n_threads
os.environ["OPENBLAS_NUM_THREADS"] = n_threads
os.environ["NUMEXPR_NUM_THREADS"] = n_threads

In [ ]:
def run_prscs_single_chrom(chrom, path_to_prscs, ref_dir, summary_stats, n_gwas, output_dir):

    output_prs = os.path.join(output_dir, f"prs_chr{chrom}")
    
    bim_path = "/genotype_data_path/merged_genome.bim" # replace your own data path
    
    command = f"python {path_to_prscs} --ref_dir {ref_dir} --bim_file {bim_path} --sst_file {summary_stats} --n_gwas {n_gwas} --chrom={chrom} --out {output_prs}"
    subprocess.run(command, shell=True, check=True) 
    print(f"Chromosome {chrom}: PRScs completed successfully.")

In [ ]:
import os
import glob

def run_prscs(PRScs_path, gwas_file, n_gwas, ref_dir, root_dir, output_prefix, njobs = 1):

    output_dir = root_dir + f"/{output_prefix}"
    os.makedirs(output_dir, exist_ok=True)
    
    chrom_lst = list(range(1, 23))  # Chromosomes 1 to 22 for autosomes
    
    cur_args = [(chrom, PRScs_path, ref_dir, gwas_file, n_gwas, output_dir) for chrom in chrom_lst]
    
    with WorkerPool(n_jobs=njobs) as pool:
        pool.map(run_prscs_single_chrom, cur_args, progress_bar=False)
    
    print(f"All chromosomes have been processed.")

In [ ]:
stats2n_gwas = {'T1_anx2026_b37_beta_p.csv': 851685,
 'T1_bip2025_b3738_beta_p.csv': 458440,
 'T1_ed2025_b3738_beta_p.csv': 458440,
 'T1_id2022_b3738_beta_p.csv': 64641,
 'T1_mdd2025_b37_beta_p.csv': 5053033,
 'T1_ocd2025_b37_beta_p.csv': 1011601,
 'T1_ptsd2024_b37_beta_p.csv': 1280933,
 'T1_scz2025_beta_p.csv': 458440,
 'T1_sud2023_b37_beta_p.csv': 1025550,
 'T2_extraversion2015_b37_beta_p.csv': 63030,
 'T2_loneliness2025_b3738_beta_p.csv': 413936,
 'T2_mtag2025_beta_p.csv': 848919,
 'T2_neuroticism2025_b3738_beta_p.csv': 279645,
 'T2_rtb2023_b3738_beta_p.csv': 274537,
 'T2_swb2024_b3738_beta_p.csv': 358135,
 'T3_cad2025_multi_b3738_beta_p.csv': 429407,
 'T3_chronic_pain_2024_multi_b3738_beta_p.csv': 576720,
 'T3_insomnia2024_multi_b3738_beta_p.csv': 575155,
 'T3_migraine2025_eur_B3738_beta_p.csv': 458440,
 'T3_t2d2025_b3738_beta_p.csv': 418949,
 'T4_bmi2025_b3738_beta_p.csv': 204747,
 'T4_crp2025_multi_b3738_beta_p.csv': 408491,
 'T5_adhd2025_b3738_beta_p.csv': 299017,
 'T5_asd2019_b37_beta_p.csv': 11178,
 'T5_intelligence2025_multi_beta_p.csv': 139298,
 'T5_neuro2025_eur_beta_p.csv': 70560,
 'T5_ts2019_b37_beta_p.csv': 14307,
 'T6_SSRI_swithcing_beta_p.csv': 21228,
 'T6_antidepressant_response_beta_p.csv': 5151,
 'T6_lithium_response_beta_p.csv': 2563
}

In [ ]:
ld_ref = "/LD_reference_path/ldblk_ukbb_eur" # replace this path by your LD reference path

PRScs_path = "./PRScs.py"

stats_lst = glob.glob("/GWAS_summary_path/PRScs_format/*.csv") # replace this by your own gwas summary path, 
                                                                # the gwas summary are saved in csv file under this path 

filenames = [os.path.basename(f) for f in stats_lst]


In [ ]:
root_dir = f"/results_path/PRScs" # replace this path by your results path

for gwas_file in stats_lst:
    stats_name = gwas_file.split("/")[-1]
    n_gwas = stats2n_gwas[stats_name]
    run_prscs(PRScs_path, gwas_file, n_gwas, ld_ref, root_dir, stats_name, njobs = 4) # change n_threads value with your computational resource, n_cpus = n_threads * njobs